In [ ]:
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

REQS = {
    "transformers": "4.51.3",
    "tokenizers": "0.21.1",
    "datasets": "3.6.0",
    "accelerate": "1.1.1",
    "huggingface-hub": "0.35.3",
    "safetensors": "0.4.3",
}

def get_ver(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

to_install = []
for pkg, min_ver in REQS.items():
    cur = get_ver(pkg)
    if cur is None or Version(cur) < Version(min_ver):
        to_install.append(f"{pkg}>={min_ver}")

if to_install:
    print("Установка:", ", ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *to_install])
else:
    print("Все нужные пакеты уже установлены.")

import torch, transformers, tokenizers, huggingface_hub
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import transformers, tokenizers, huggingface_hub

from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    matthews_corrcoef
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments,
    Trainer, EarlyStoppingCallback
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "microsoft/deberta-v3-large"
MAX_LENGTH = 128

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# Пути к файлам (Colab/Kaggle/локально)
CRYPTO_PATH = "crypto_twitter_dataset2.csv"
NON_CRYPTO_PATH = "non_crypto_tweets.csv"

SEARCH_ROOTS = [Path("."), Path("/kaggle/input"), Path("/kaggle/working")]

def find_file_by_name(filename):
    p = Path(filename)
    if p.exists():
        return p

    for root in SEARCH_ROOTS:
        if root.exists():
            hits = list(root.rglob(p.name))
            if hits:
                return hits[0]

    key = p.stem.lower().replace("-", "").replace("_", "")
    for root in SEARCH_ROOTS:
        if root.exists():
            for f in root.rglob("*.csv"):
                stem_norm = f.stem.lower().replace("-", "").replace("_", "")
                if key in stem_norm or stem_norm in key:
                    return f

    return None

def safe_read_csv(path):
    found = find_file_by_name(path)
    if found is None:
        roots = [str(r) for r in SEARCH_ROOTS if r.exists()]
        raise FileNotFoundError(
            f"Не найден файл: {path}.{roots}"
        )

    try:
        df = pd.read_csv(
            found,
            sep=';',
            engine="python",
            on_bad_lines="skip",
            quotechar='"',
            encoding="utf-8"
        )
    except Exception:
        df = pd.read_csv(
            found,
            sep=None,
            engine="python",
            on_bad_lines="skip",
            quotechar='"',
            encoding="utf-8"
        )

    if df.empty:
        raise ValueError(f"Файл {found} прочитан.")

    print(f"Loaded: {found}")
    return df

crypto_df = safe_read_csv(CRYPTO_PATH)
non_crypto_df = safe_read_csv(NON_CRYPTO_PATH)

print("Crypto shape:", crypto_df.shape)
print("Non-crypto shape:", non_crypto_df.shape)
print("Crypto columns:", list(crypto_df.columns)[:10], "...")
print("Non-crypto columns:", list(non_crypto_df.columns))

In [ ]:
TEXT_CANDIDATES = ["tweet_text", "full_text", "text", "tweet"]
ID_CANDIDATES = ["tweet_id", "id"]

def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_df(df, label):
    text_col = first_existing_column(df, TEXT_CANDIDATES)
    if text_col is None:
        raise ValueError(f"Не найден текстовый столбец. Ожидались: {TEXT_CANDIDATES}")

    id_col = first_existing_column(df, ID_CANDIDATES)

    out = pd.DataFrame({
        "text": df[text_col].astype(str),
        "label": int(label)
    })
    if id_col is not None:
        out["tweet_id"] = df[id_col].astype(str)
    else:
        out["tweet_id"] = np.arange(len(out)).astype(str)
    return out

crypto_norm = normalize_df(crypto_df, label=1)
non_crypto_norm = normalize_df(non_crypto_df, label=0)
data = pd.concat([crypto_norm, non_crypto_norm], ignore_index=True)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data["text"] = data["text"].fillna("").map(clean_text)

before = len(data)
data = data[data["text"].str.len() >= 8]
data = data[data["text"].str.len() <= 800]
data = data[data["text"].str.contains(r"[\w\d]", regex=True)]
data = data.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
after = len(data)

text_len = data["text"].str.len()
dup_text_rate = data.duplicated(subset=["text"]).mean()
class_dist = data["label"].value_counts().sort_index()

print(f"Rows before clean: {before}")
print(f"Rows after clean:  {after}")
print(f"Dropped:           {before - after}")
print("Class counts:")
print(class_dist)
print("Class ratio (0/1):", round(class_dist.get(0, 0) / max(class_dist.get(1, 1), 1), 3))
print(f"Duplicate text rate (ignoring label): {dup_text_rate:.4f}")
print("Text length stats:")
print(text_len.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("\nSample rows:")
print(data.head(3))

In [ ]:
# Train / validation / test split (stratified)
temp_test_size = 0.10  
val_share_in_temp = 0.50

train_df, temp_df = train_test_split(
    data, test_size=temp_test_size, random_state=SEED, stratify=data["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=1 - val_share_in_temp, random_state=SEED, stratify=temp_df["label"]
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

for split_name, ds_obj in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    drop_cols = [c for c in ds_obj.column_names if c not in {"input_ids", "attention_mask", "token_type_ids", "label"}]
    if drop_cols:
        if split_name == "train":
            train_ds = train_ds.remove_columns(drop_cols)
        elif split_name == "val":
            val_ds = val_ds.remove_columns(drop_cols)
        else:
            test_ds = test_ds.remove_columns(drop_cols)

train_ds = train_ds.filter(lambda x: len(x["input_ids"]) >= 3)
val_ds = val_ds.filter(lambda x: len(x["input_ids"]) >= 3)
test_ds = test_ds.filter(lambda x: len(x["input_ids"]) >= 3)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    logits = np.asarray(logits)
    logits = np.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0)

    probs = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)
    preds = np.argmax(probs, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    try:
        roc_auc = roc_auc_score(labels, probs[:, 1])
    except Exception:
        roc_auc = float("nan")

    try:
        pr_auc = average_precision_score(labels, probs[:, 1])
    except Exception:
        pr_auc = float("nan")

    try:
        mcc = matthews_corrcoef(labels, preds)
    except Exception:
        mcc = float("nan")

    pred_pos_rate = float(np.mean(preds == 1))
    true_pos_rate = float(np.mean(np.asarray(labels) == 1))

    return {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc) if not np.isnan(roc_auc) else 0.0,
        "pr_auc": float(pr_auc) if not np.isnan(pr_auc) else 0.0,
        "mcc": float(mcc) if not np.isnan(mcc) else 0.0,
        "pred_pos_rate": pred_pos_rate,
        "true_pos_rate": true_pos_rate
    }

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU не активен в PyTorch. Включите GPU и перезапустите kernel."
    )

import gc
gc.collect()
torch.cuda.empty_cache()

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)

use_fp16 = any(x in gpu_name for x in ["T4", "L4", "A10", "A100", "V100"])
if any(x in gpu_name for x in ["P100", "K80"]):
    use_fp16 = False
print("fp16 enabled:", use_fp16)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    use_safetensors=False,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
).to(device)

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

for p in model.parameters():
    p.requires_grad = False

for name, p in model.named_parameters():
    if name.startswith("classifier") or name.startswith("pooler") or name.startswith("deberta.pooler"):
        p.requires_grad = True
    elif "encoder.layer." in name:
        try:
            layer_id = int(name.split("encoder.layer.")[1].split(".")[0])
            if layer_id >= 16:
                p.requires_grad = True
        except Exception:
            pass

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_fast = train_ds
val_fast = val_ds

train_counts = np.bincount(np.array(train_fast["label"]), minlength=2)
val_counts = np.bincount(np.array(val_fast["label"]), minlength=2)
print(f"Train set: {len(train_fast)} | 0={train_counts[0]}, 1={train_counts[1]}")
print(f"Val set:   {len(val_fast)} | 0={val_counts[0]}, 1={val_counts[1]}")
print(f"Train positive rate: {train_counts[1] / max(1, train_counts.sum()):.4f}")
print(f"Val positive rate:   {val_counts[1] / max(1, val_counts.sum()):.4f}")

probe = data_collator([train_fast[i] for i in range(min(8, len(train_fast)))])
if "label" in probe and "labels" not in probe:
    probe["labels"] = probe.pop("label")
probe = {k: v.to(device) for k, v in probe.items()}
model.train()
probe_out = model(**probe)
print("Probe loss finite:", bool(torch.isfinite(probe_out.loss).item()))

per_device_train_bs = 2 if use_fp16 else 1
per_device_eval_bs = 8 if use_fp16 else 2
learning_rate = 8e-6

training_args = TrainingArguments(
    output_dir="./deberta_crypto_detector_stable_quality_plus",
    learning_rate=learning_rate,
    num_train_epochs=4,
    per_device_train_batch_size=per_device_train_bs,
    per_device_eval_batch_size=per_device_eval_bs,
    gradient_accumulation_steps=1,
    warmup_ratio=0.10,
    weight_decay=0.01,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    logging_nan_inf_filter=True,
    report_to="none",
    fp16=use_fp16,
    bf16=False,
    gradient_checkpointing=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    group_by_length=True,
    optim="adamw_torch",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_fast,
    eval_dataset=val_fast,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()
print("Stable quality+ training finished successfully.")

In [ ]:
test_metrics = trainer.evaluate(test_ds)
print("Test metrics:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

PREDICT_MAX_SAMPLES = 2500
if len(test_ds) > PREDICT_MAX_SAMPLES:
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(len(test_ds), size=PREDICT_MAX_SAMPLES, replace=False).tolist()
    test_pred_ds = test_ds.select(sample_idx)
    print(f"Using sampled test subset for detailed report: {len(test_pred_ds)} / {len(test_ds)}")
else:
    test_pred_ds = test_ds

pred_out = trainer.predict(test_pred_ds)
logits = pred_out.predictions
y_true = pred_out.label_ids
y_prob = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
y_pred = np.argmax(logits, axis=-1)

print("\nClassification report:")
print(classification_report(y_true, y_pred, digits=4, target_names=["non_crypto", "crypto"]))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
labels = ["non_crypto", "crypto"]

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(labels))
plt.xticks(tick_marks, labels)
plt.yticks(tick_marks, labels)
plt.xlabel("Predicted")
plt.ylabel("True")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

plt.tight_layout()
plt.show()

In [ ]:
SAVE_DIR = "./deberta_crypto_detector_best"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

In [ ]:
id2label = {0: "non_crypto", 1: "crypto"}

def predict_tweet(text, model, tokenizer, max_length=128):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    if torch.cuda.is_available():
        enc = {k: v.cuda() for k, v in enc.items()}
        model.cuda()

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred = int(np.argmax(probs))
    return {
        "label": id2label[pred],
        "crypto_probability": float(probs[1]),
        "non_crypto_probability": float(probs[0])
    }

sample = "Bitcoin ETF inflows hit a new high this week, market looks bullish."
print(predict_tweet(sample, model, tokenizer))